In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5-20251001"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [4]:
dataset = generate_dataset()
dataset

[{'task': "Write a Python function that extracts the AWS region from an S3 bucket URL (e.g., 's3.us-west-2.amazonaws.com'). Return the region code as a string."},
 {'task': "Create a JSON object that represents an AWS IAM policy allowing read-only access to a specific S3 bucket named 'my-bucket'. Include the necessary Action, Resource, and Effect fields."},
 {'task': "Write a regular expression that matches valid AWS EC2 instance IDs. EC2 instance IDs follow the pattern: 'i-' followed by 17 hexadecimal characters (e.g., 'i-0a1b2c3d4e5f6g7h8')."}]

In [6]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [8]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [9]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [10]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Region Extraction Function\n\nHere's a Python function that extracts the AWS region from an S3 bucket URL:\n\n```python\nimport re\n\ndef extract_s3_region(s3_url):\n    \"\"\"\n    Extracts the AWS region from an S3 bucket URL.\n    \n    Args:\n        s3_url (str): S3 URL in formats like:\n            - 's3.us-west-2.amazonaws.com'\n            - 'https://s3.us-west-2.amazonaws.com'\n            - 'bucket-name.s3.us-west-2.amazonaws.com'\n            - 's3-us-west-2.amazonaws.com'\n    \n    Returns:\n        str: The region code (e.g., 'us-west-2'), or None if not found\n    \"\"\"\n    # Pattern 1: s3.REGION.amazonaws.com or s3-REGION.amazonaws.com\n    pattern1 = r's3[.-]([a-z0-9\\-]+)\\.amazonaws\\.com'\n    match = re.search(pattern1, s3_url)\n    if match:\n        return match.group(1)\n    \n    # Pattern 2: bucket.s3.REGION.amazonaws.com\n    pattern2 = r's3\\.([a-z0-9\\-]+)\\.amazonaws\\.com'\n    match = re.search(pattern2, s3_url)\n    if ma